# Dice Detector 🎲 — Kaggle Training v3 (RT-DETRv2 + value head)

This version replaces the YOLO multi-head detector (v1) with a **transformer detector**:

- **Backbone / detector**: `RTDetrV2ForObjectDetection` (`PekingU/rtdetr_v2_r18vd`).
  Die type is the **native detection class** (7 classes: `D4, D6, D8, D10, D12, D20, D100`),
  so we get sigmoid focal loss + **CDN (contrastive denoising)** + **auxiliary deep supervision**
  for free just by passing `labels=`.
- **Value head** (the only custom part): an MLP on `last_hidden_state` predicting the
  face value (21 classes, `0..20`). Supervised on matched queries, **masked** when the
  value is unknown / unreadable.
- **D100 handling**: D100 is visually a D10 (digits `0..9`), so its value is encoded as
  `value // 10` (i.e. `0,10,...,90` → class `0..9`). The percentile roll is reconstructed
  by pairing a `D10` + `D100` at detection time.

Runs on Kaggle (CUDA T4/P100) **and** locally on AMD via ROCm — the `"cuda"` device checks
work transparently under HIP, so an RX 9070 XT (16 GB) handles `r18vd` at 640px easily.

## 1. Setup

In [ ]:
!pip install -q "transformers>=4.44" torchmetrics scipy pillow

In [ ]:
from __future__ import annotations

import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch

# ----- Kaggle paths -----
KAGGLE_DATASET_DIR = Path('/kaggle/input/datasets/oskarwichtowski/d-and-d-dices-dataset')
WORK_DIR = Path('/kaggle/working')
OUTPUT_CKPT = WORK_DIR / 'dice_rtdetr.pt'

# ----- Model / training config -----
PRETRAINED_MODEL = 'PekingU/rtdetr_v2_r18vd'
IMAGE_SIZE = 640
EPOCHS = 100                 # ceiling, not a target — early stopping ends it sooner
EARLY_STOP_PATIENCE = 12     # stop after this many epochs with no mAP improvement
BATCH = 4
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
VALUE_LOSS_WEIGHT = 1.0
USE_AMP = True

# ----- Label spaces -----
DIE_TYPES = ['D4', 'D6', 'D8', 'D10', 'D12', 'D20', 'D100']
DIE_TYPE_TO_ID = {dt: i for i, dt in enumerate(DIE_TYPES)}
NUM_DIE_TYPES = len(DIE_TYPES)

VALUE_LABELS = [str(i) for i in range(0, 21)]   # 0..20
NUM_VALUE_CLASSES = len(VALUE_LABELS)           # 21

TRAIN_RATIO, VAL_RATIO = 0.80, 0.15
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={device} | dataset exists={KAGGLE_DATASET_DIR.exists()}')
print(f'die types={NUM_DIE_TYPES} | value classes={NUM_VALUE_CLASSES}')


def value_to_class(dice_type: str, value) -> int:
    """Synthetic value -> value-class 0..20, or -1 (masked/unknown).
    D100 faces show 0,10,..,90 -> class = value // 10 (0..9)."""
    if value is None:
        return -1
    try:
        v = int(value)
    except (TypeError, ValueError):
        return -1
    vc = v // 10 if dice_type == 'D100' else v
    return vc if 0 <= vc < NUM_VALUE_CLASSES else -1


def class_to_value(dice_type: str, value_class: int) -> int:
    """Inverse of value_to_class for display."""
    return int(value_class) * 10 if dice_type == 'D100' else int(value_class)

## 2. Model — RT-DETRv2 + value head

Die type is the native detection class; the value head is a small MLP on the decoder's
`last_hidden_state`. Pass `labels=` during training to activate the native RT-DETR loss
(focal + L1 + GIoU + CDN + aux).

In [ ]:
import torch.nn as nn
from torchvision.ops import box_convert
from transformers import RTDetrImageProcessor, RTDetrV2ForObjectDetection


def make_processor(pretrained_model: str, image_size: int = 640):
    """Same processor MUST be used in training and inference.
    RTDetrImageProcessor stretches to size (no letterbox/padding)."""
    return RTDetrImageProcessor.from_pretrained(
        pretrained_model,
        size={'height': image_size, 'width': image_size},
        do_resize=True,
        do_rescale=True,
        do_normalize=True,
        do_pad=False,
    )


def xyxy_abs_to_cxcywh_norm(boxes_xyxy_abs, width, height):
    if boxes_xyxy_abs.numel() == 0:
        return boxes_xyxy_abs.reshape(0, 4)
    boxes = boxes_xyxy_abs.float().clone()
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, width)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, height)
    boxes_cxcywh = box_convert(boxes, in_fmt='xyxy', out_fmt='cxcywh')
    scale = torch.tensor([width, height, width, height], dtype=torch.float32)
    return boxes_cxcywh / scale


def cxcywh_norm_to_xyxy_abs(boxes_cxcywh_norm, width, height):
    if boxes_cxcywh_norm.numel() == 0:
        return boxes_cxcywh_norm.reshape(0, 4)
    boxes_xyxy = box_convert(boxes_cxcywh_norm, in_fmt='cxcywh', out_fmt='xyxy')
    scale = torch.tensor([width, height, width, height], dtype=torch.float32,
                         device=boxes_xyxy.device)
    boxes_xyxy = boxes_xyxy * scale
    boxes_xyxy[:, [0, 2]] = boxes_xyxy[:, [0, 2]].clamp(0, width)
    boxes_xyxy[:, [1, 3]] = boxes_xyxy[:, [1, 3]].clamp(0, height)
    return boxes_xyxy


def get_hidden_dim(config):
    for name in ['d_model', 'hidden_size', 'hidden_dim']:
        if hasattr(config, name):
            return int(getattr(config, name))
    return 256


class DiceRTDETR(nn.Module):
    def __init__(self, num_value_classes=NUM_VALUE_CLASSES,
                 pretrained_model=PRETRAINED_MODEL, dropout=0.1):
        super().__init__()
        self.pretrained_model = pretrained_model
        self.num_die_types = len(DIE_TYPES)
        id2label = {i: dt for i, dt in enumerate(DIE_TYPES)}
        label2id = {dt: i for i, dt in enumerate(DIE_TYPES)}
        self.detector = RTDetrV2ForObjectDetection.from_pretrained(
            pretrained_model,
            num_labels=self.num_die_types,
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )
        hidden_dim = get_hidden_dim(self.detector.config)
        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_value_classes),
        )

    def forward(self, pixel_values, pixel_mask=None, labels=None):
        outputs = self.detector(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            labels=labels,
            output_hidden_states=True,
            return_dict=True,
        )
        hidden = outputs.last_hidden_state
        return {
            'det_logits': outputs.logits,
            'pred_boxes': outputs.pred_boxes,
            'value_logits': self.value_head(hidden),
            'det_loss': outputs.loss,
        }


def save_checkpoint(path, model, optimizer, epoch, image_size, best_val_loss=None):
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict() if optimizer is not None else None,
        'epoch': epoch,
        'pretrained_model': model.pretrained_model,
        'image_size': image_size,
        'die_types': list(DIE_TYPES),
        'value_labels': list(VALUE_LABELS),
        'best_val_loss': best_val_loss,
    }, path)


def load_checkpoint(path, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = DiceRTDETR(num_value_classes=len(ckpt['value_labels']),
                       pretrained_model=ckpt['pretrained_model'])
    model.load_state_dict(ckpt['model_state'])
    model.to(device)
    model.eval()
    return model, ckpt


processor = make_processor(PRETRAINED_MODEL, IMAGE_SIZE)
print('processor ready')

## 3. Dataset

Reads the synthetic per-image JSON (`image_width`, `image_height`, `dice: [{dice_type,
value, bbox:{x,y,width,height}}]`), converts bboxes to normalized `cxcywh`, and maps
values to value-classes. Unknown values become `-1` (masked from the value loss).
Images are split by **render group** so frames from the same scene don't leak across
train/val/test. Preprocessing runs inside `__getitem__` so `num_workers` parallelizes it.

In [ ]:
import re

from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader

ImageFile.LOAD_TRUNCATED_IMAGES = True


def extract_render_group(filename: str) -> str:
    m = re.search(r'render_(\d+)', filename)
    return str(int(m.group(1)) // 100) if m else filename


img_dir = KAGGLE_DATASET_DIR / 'images'
ann_dir = KAGGLE_DATASET_DIR / 'annotations'
ann_files = sorted(ann_dir.glob('*.json'))
pairs = [(img_dir / f'{a.stem}.png', a) for a in ann_files
         if (img_dir / f'{a.stem}.png').exists()]
print(f'{len(pairs)} image+annotation pairs')

groups = defaultdict(list)
for pair in pairs:
    groups[extract_render_group(pair[0].stem)].append(pair)
group_keys = list(groups.keys())
random.shuffle(group_keys)
n = len(group_keys)
n_train, n_val = int(n * TRAIN_RATIO), int(n * VAL_RATIO)
split_groups = {
    'train': group_keys[:n_train],
    'val': group_keys[n_train:n_train + n_val],
    'test': group_keys[n_train + n_val:],
}
splits = {k: [p for g in gs for p in groups[g]] for k, gs in split_groups.items()}
for sp in splits.values():
    random.shuffle(sp)
print('Split:', {k: len(v) for k, v in splits.items()})


class DiceDataset(Dataset):
    def __init__(self, pairs, processor):
        self.pairs = pairs
        self.processor = processor

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, ann_path = self.pairs[idx]
        image = Image.open(img_path).convert('RGB')
        width, height = image.size

        encoded = self.processor(images=[image], return_tensors='pt')
        pixel_values = encoded['pixel_values'].squeeze(0)

        with open(ann_path) as f:
            data = json.load(f)

        boxes_xyxy, type_ids, value_ids = [], [], []
        for die in data.get('dice', []):
            dtype = die.get('dice_type')
            if dtype not in DIE_TYPE_TO_ID:
                continue
            b = die['bbox']
            x1, y1 = b['x'], b['y']
            x2, y2 = x1 + b['width'], y1 + b['height']
            boxes_xyxy.append([x1, y1, x2, y2])
            type_ids.append(DIE_TYPE_TO_ID[dtype])
            value_ids.append(value_to_class(dtype, die.get('value')))

        if boxes_xyxy:
            boxes_t = torch.tensor(boxes_xyxy, dtype=torch.float32)
            boxes = xyxy_abs_to_cxcywh_norm(boxes_t, width, height)
            class_labels = torch.tensor(type_ids, dtype=torch.long)
            value_labels = torch.tensor(value_ids, dtype=torch.long)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            class_labels = torch.zeros((0,), dtype=torch.long)
            value_labels = torch.zeros((0,), dtype=torch.long)

        target = {
            'boxes': boxes,
            'class_labels': class_labels,
            'value_labels': value_labels,
        }
        return pixel_values, target


def collate_fn(batch):
    pixel_values_list, targets = zip(*batch)
    return {
        'pixel_values': torch.stack(pixel_values_list, dim=0),
        'targets': list(targets),
    }


def move_targets_to_device(targets, device):
    return [{
        'boxes': t['boxes'].to(device),
        'class_labels': t['class_labels'].to(device),
        'value_labels': t['value_labels'].to(device),
    } for t in targets]


train_loader = DataLoader(DiceDataset(splits['train'], processor), batch_size=BATCH,
                          shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn,
                          pin_memory=device.type == 'cuda',
                          persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(DiceDataset(splits['val'], processor), batch_size=BATCH,
                        shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn,
                        pin_memory=device.type == 'cuda',
                        persistent_workers=NUM_WORKERS > 0)
print('loaders ready:', len(train_loader), 'train batches,', len(val_loader), 'val batches')

## 4. Sanity check — visualize a few samples

In [ ]:
from IPython.display import display
from PIL import ImageDraw, ImageFont

TYPE_COLORS = {
    'D4': '#FF6B6B', 'D6': '#4ECDC4', 'D8': '#45B7D1', 'D10': '#96CEB4',
    'D12': '#FFEAA7', 'D20': '#DDA0DD', 'D100': '#98D8C8',
}


def _font(size=14):
    try:
        return ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', size)
    except OSError:
        return ImageFont.load_default()


for img_path, ann_path in random.sample(splits['train'], min(3, len(splits['train']))):
    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    font = _font()
    with open(ann_path) as f:
        data = json.load(f)
    for die in data.get('dice', []):
        dtype = die.get('dice_type')
        if dtype not in DIE_TYPE_TO_ID:
            continue
        b = die['bbox']
        x1, y1, x2, y2 = b['x'], b['y'], b['x'] + b['width'], b['y'] + b['height']
        color = TYPE_COLORS.get(dtype, '#FFFFFF')
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        vc = value_to_class(dtype, die.get('value'))
        label = f'{dtype}:{class_to_value(dtype, vc) if vc >= 0 else "?"}'
        draw.text((x1, y1 - 16), label, fill=color, font=font)
    img.thumbnail((512, 512))
    display(img)

## 5. Training

Detection loss is computed natively by RT-DETR (`labels=`). The **value head** uses its
own lightweight Hungarian matching on boxes (L1 + GIoU, fp32) to assign queries to GT,
then a masked cross-entropy. Validation reports **mAP**, **mAP@50**, and per-head
**type / value accuracy**.

In [ ]:
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import generalized_box_iou
from tqdm.auto import tqdm


@torch.no_grad()
def match_for_value_head(pred_boxes, targets):
    """Box-only Hungarian matching for value-head supervision (fp32)."""
    indices = []
    for b in range(pred_boxes.shape[0]):
        tgt_boxes = targets[b]['boxes']
        if tgt_boxes.shape[0] == 0:
            empty = torch.empty(0, dtype=torch.long, device=pred_boxes.device)
            indices.append((empty, empty))
            continue
        pred_b = pred_boxes[b].float()
        tgt_b = tgt_boxes.float()
        cost_bbox = torch.cdist(pred_b, tgt_b, p=1)
        pred_xyxy = box_convert(pred_b, in_fmt='cxcywh', out_fmt='xyxy')
        tgt_xyxy = box_convert(tgt_b, in_fmt='cxcywh', out_fmt='xyxy')
        cost_giou = -generalized_box_iou(pred_xyxy, tgt_xyxy)
        final_cost = 5.0 * cost_bbox + 2.0 * cost_giou
        src_idx, tgt_idx = linear_sum_assignment(final_cost.detach().cpu().numpy())
        indices.append((
            torch.as_tensor(src_idx, dtype=torch.long, device=pred_boxes.device),
            torch.as_tensor(tgt_idx, dtype=torch.long, device=pred_boxes.device),
        ))
    return indices


def compute_value_loss(outputs, targets, indices):
    value_logits = outputs['value_logits']
    matched_logits, matched_targets = [], []
    for b, (src_idx, tgt_idx) in enumerate(indices):
        if len(src_idx) == 0:
            continue
        tgt_values = targets[b]['value_labels'][tgt_idx]
        valid = tgt_values >= 0
        if valid.sum() == 0:
            continue
        matched_logits.append(value_logits[b, src_idx[valid]])
        matched_targets.append(tgt_values[valid])
    if not matched_logits:
        return value_logits.sum() * 0.0
    return F.cross_entropy(torch.cat(matched_logits), torch.cat(matched_targets))


def train_one_epoch(model, loader, optimizer, device, value_loss_weight, use_amp):
    model.train()
    running = {'loss_total': 0.0, 'loss_det': 0.0, 'loss_value': 0.0}
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp and device.type == 'cuda')
    for batch in tqdm(loader, desc='train', leave=False):
        pixel_values = batch['pixel_values'].to(device)
        targets = move_targets_to_device(batch['targets'], device)
        det_labels = [{'class_labels': t['class_labels'], 'boxes': t['boxes']} for t in targets]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type,
                                enabled=use_amp and device.type == 'cuda'):
            outputs = model(pixel_values, labels=det_labels)
            loss_det = outputs['det_loss']
            with torch.amp.autocast(device_type=device.type, enabled=False):
                indices = match_for_value_head(outputs['pred_boxes'].float(), targets)
                loss_value = compute_value_loss(outputs, targets, indices)
            loss = loss_det + value_loss_weight * loss_value
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running['loss_total'] += float(loss.detach().cpu())
        running['loss_det'] += float(loss_det.detach().cpu())
        running['loss_value'] += float(loss_value.detach().cpu())
    nb = max(len(loader), 1)
    return {k: v / nb for k, v in running.items()}


@torch.no_grad()
def validate(model, loader, device, value_loss_weight):
    model.eval()
    running = {'loss_total': 0.0, 'loss_det': 0.0, 'loss_value': 0.0}
    map_metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    type_correct = type_total = value_correct = value_total = 0
    for batch in tqdm(loader, desc='val', leave=False):
        pixel_values = batch['pixel_values'].to(device)
        targets = move_targets_to_device(batch['targets'], device)
        det_labels = [{'class_labels': t['class_labels'], 'boxes': t['boxes']} for t in targets]
        outputs = model(pixel_values, labels=det_labels)
        loss_det = outputs['det_loss']
        indices = match_for_value_head(outputs['pred_boxes'].float(), targets)
        loss_value = compute_value_loss(outputs, targets, indices)
        loss = loss_det + value_loss_weight * loss_value
        running['loss_total'] += float(loss.detach().cpu())
        running['loss_det'] += float(loss_det.detach().cpu())
        running['loss_value'] += float(loss_value.detach().cpu())
        det_logits = outputs['det_logits']
        pred_boxes = outputs['pred_boxes']
        value_logits = outputs['value_logits']
        for b in range(det_logits.shape[0]):
            scores = det_logits[b].sigmoid()
            max_scores, pred_classes = scores.max(-1)
            keep = max_scores > 0.3
            if keep.sum() > 0:
                pred_xyxy = box_convert(pred_boxes[b][keep], in_fmt='cxcywh', out_fmt='xyxy')
                preds_for_map = [{'boxes': pred_xyxy.cpu(), 'scores': max_scores[keep].cpu(),
                                  'labels': pred_classes[keep].cpu()}]
            else:
                preds_for_map = [{'boxes': torch.zeros((0, 4)), 'scores': torch.zeros(0),
                                  'labels': torch.zeros(0, dtype=torch.long)}]
            tgt_xyxy = box_convert(targets[b]['boxes'], in_fmt='cxcywh', out_fmt='xyxy')
            targets_for_map = [{'boxes': tgt_xyxy.cpu(), 'labels': targets[b]['class_labels'].cpu()}]
            map_metric.update(preds_for_map, targets_for_map)
            src_idx, tgt_idx = indices[b]
            if len(src_idx) == 0:
                continue
            pred_types = pred_classes[src_idx]
            gt_types = targets[b]['class_labels'][tgt_idx]
            type_correct += int((pred_types == gt_types).sum())
            type_total += len(src_idx)
            gt_values = targets[b]['value_labels'][tgt_idx]
            valid = gt_values >= 0
            if valid.sum() > 0:
                pred_values = value_logits[b, src_idx[valid]].argmax(-1)
                value_correct += int((pred_values == gt_values[valid]).sum())
                value_total += int(valid.sum())
    nb = max(len(loader), 1)
    metrics = {k: v / nb for k, v in running.items()}
    map_results = map_metric.compute()
    metrics['mAP'] = float(map_results['map'])
    metrics['mAP_50'] = float(map_results['map_50'])
    metrics['type_accuracy'] = type_correct / max(type_total, 1)
    metrics['value_accuracy'] = value_correct / max(value_total, 1)
    return metrics

In [ ]:
import copy

import matplotlib.pyplot as plt
from IPython.display import clear_output


def plot_history(history):
    """Redraw the live training dashboard from accumulated per-epoch history."""
    epochs = history['epoch']
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    # Total loss (train vs val)
    ax = axes[0, 0]
    ax.plot(epochs, history['train_total'], '-o', label='train', markersize=3)
    ax.plot(epochs, history['val_total'], '-o', label='val', markersize=3)
    ax.set_title('Total loss')
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); ax.grid(alpha=0.3)

    # Loss components (val)
    ax = axes[0, 1]
    ax.plot(epochs, history['val_det'], '-o', label='det', markersize=3)
    ax.plot(epochs, history['val_value'], '-o', label='value', markersize=3)
    ax.set_title('Val loss components')
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); ax.grid(alpha=0.3)

    # mAP
    ax = axes[1, 0]
    ax.plot(epochs, history['mAP'], '-o', label='mAP', markersize=3)
    ax.plot(epochs, history['mAP_50'], '-o', label='mAP@50', markersize=3)
    ax.set_title('Detection mAP')
    ax.set_xlabel('epoch'); ax.set_ylabel('mAP'); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=0.3)

    # Per-head accuracy
    ax = axes[1, 1]
    ax.plot(epochs, history['type_accuracy'], '-o', label='type acc', markersize=3)
    ax.plot(epochs, history['value_accuracy'], '-o', label='value acc', markersize=3)
    ax.set_title('Per-head accuracy')
    ax.set_xlabel('epoch'); ax.set_ylabel('accuracy'); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=0.3)

    fig.tight_layout()
    plt.show()


model = DiceRTDETR(num_value_classes=NUM_VALUE_CLASSES,
                   pretrained_model=PRETRAINED_MODEL).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = {k: [] for k in ['epoch', 'train_total', 'val_total', 'val_det', 'val_value',
                           'mAP', 'mAP_50', 'type_accuracy', 'value_accuracy']}

best_map = -1.0              # early stopping monitors mAP (higher is better)
best_epoch = 0
best_state = None
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    tm = train_one_epoch(model, train_loader, optimizer, device, VALUE_LOSS_WEIGHT, USE_AMP)
    vm = validate(model, val_loader, device, VALUE_LOSS_WEIGHT)

    history['epoch'].append(epoch)
    history['train_total'].append(tm['loss_total'])
    history['val_total'].append(vm['loss_total'])
    history['val_det'].append(vm['loss_det'])
    history['val_value'].append(vm['loss_value'])
    history['mAP'].append(vm['mAP'])
    history['mAP_50'].append(vm['mAP_50'])
    history['type_accuracy'].append(vm['type_accuracy'])
    history['value_accuracy'].append(vm['value_accuracy'])

    # Live update: clear previous output and redraw the dashboard + log line
    clear_output(wait=True)
    plot_history(history)
    print(f"epoch={epoch} train_total={tm['loss_total']:.4f} "
          f"det={tm['loss_det']:.4f} value={tm['loss_value']:.4f}")
    print(f"epoch={epoch} val_total={vm['loss_total']:.4f} det={vm['loss_det']:.4f} "
          f"value={vm['loss_value']:.4f} mAP={vm['mAP']:.4f} mAP@50={vm['mAP_50']:.4f} "
          f"type_acc={vm['type_accuracy']:.4f} value_acc={vm['value_accuracy']:.4f}")

    if vm['mAP'] > best_map:
        best_map = vm['mAP']
        best_epoch = epoch
        epochs_no_improve = 0
        best_state = copy.deepcopy(model.state_dict())
        save_checkpoint(OUTPUT_CKPT, model, optimizer, epoch, IMAGE_SIZE,
                        best_val_loss=vm['loss_total'])
        print(f'saved checkpoint: {OUTPUT_CKPT} (mAP={best_map:.4f})')
    else:
        epochs_no_improve += 1
        print(f'no mAP improvement for {epochs_no_improve}/{EARLY_STOP_PATIENCE} epochs '
              f'(best mAP={best_map:.4f} @ epoch {best_epoch})')
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f'early stopping at epoch {epoch} — best mAP={best_map:.4f} @ epoch {best_epoch}')
            break

# Restore the best weights into the in-memory model for the inference cells below
if best_state is not None:
    model.load_state_dict(best_state)
    print(f'restored best weights (mAP={best_map:.4f} @ epoch {best_epoch})')

## 6. Inference — predict on a few test images

RT-DETR uses **sigmoid** classification, so detections are selected by sigmoid score +
top-k (not softmax).

In [ ]:
@torch.no_grad()
def predict(image, model, ckpt, threshold=0.45, top_k=100):
    proc = make_processor(ckpt['pretrained_model'], ckpt['image_size'])
    die_types = ckpt['die_types']
    width, height = image.size
    encoded = proc(images=[image], return_tensors='pt')
    pixel_values = encoded['pixel_values'].to(device)
    outputs = model(pixel_values)
    det_scores = outputs['det_logits'][0].sigmoid()
    max_scores, det_classes = det_scores.max(dim=-1)
    value_probs = outputs['value_logits'][0].softmax(-1)
    boxes = outputs['pred_boxes'][0]
    keep = max_scores >= threshold
    if keep.sum() == 0:
        return []
    scores, classes = max_scores[keep], det_classes[keep]
    boxes, value_probs = boxes[keep], value_probs[keep]
    if scores.numel() > top_k:
        scores, top_idx = torch.topk(scores, k=top_k)
        classes, boxes, value_probs = classes[top_idx], boxes[top_idx], value_probs[top_idx]
    boxes_xyxy = cxcywh_norm_to_xyxy_abs(boxes, width, height)
    value_conf, value_ids = value_probs.max(dim=-1)
    results = []
    for i in range(scores.shape[0]):
        dtype = die_types[int(classes[i])]
        vc = int(value_ids[i])
        results.append({
            'bbox_xyxy': [round(float(x), 1) for x in boxes_xyxy[i].cpu().tolist()],
            'detection_score': round(float(scores[i]), 4),
            'die_type': dtype,
            'value': class_to_value(dtype, vc),
            'value_score': round(float(value_conf[i]), 4),
        })
    results.sort(key=lambda r: r['detection_score'], reverse=True)
    return results


model, ckpt = load_checkpoint(OUTPUT_CKPT, device)
for img_path, _ in random.sample(splits['test'], min(3, len(splits['test']))):
    image = Image.open(img_path).convert('RGB')
    dets = predict(image, model, ckpt, threshold=0.45)
    vis = image.copy()
    draw = ImageDraw.Draw(vis)
    font = _font()
    for d in dets:
        x1, y1, x2, y2 = d['bbox_xyxy']
        color = TYPE_COLORS.get(d['die_type'], '#FFFFFF')
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        draw.text((x1, y1 - 16), f"{d['die_type']}:{d['value']} ({d['detection_score']:.2f})",
                  fill=color, font=font)
    print(img_path.name, '->', len(dets), 'detections')
    vis.thumbnail((512, 512))
    display(vis)

## 7. Output

The best checkpoint is saved to `/kaggle/working/dice_rtdetr.pt` and will appear in the
notebook's **Output** tab after the session ends. It is self-describing (stores
`pretrained_model`, `image_size`, `die_types`, `value_labels`) so `load_checkpoint`
alone is enough to run inference elsewhere.

In [ ]:
import os

if OUTPUT_CKPT.exists():
    size_mb = os.path.getsize(OUTPUT_CKPT) / 1e6
    print(f'checkpoint: {OUTPUT_CKPT} ({size_mb:.1f} MB)')
else:
    print('no checkpoint saved yet — run the training cell first')

## 8. Zip the checkpoint

Bundle the checkpoint into a single `.zip` in `/kaggle/working` for easy download from
the **Output** tab.

In [ ]:
import zipfile

ZIP_PATH = WORK_DIR / 'dice_rtdetr.zip'

if OUTPUT_CKPT.exists():
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(OUTPUT_CKPT, arcname=OUTPUT_CKPT.name)
    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f'zipped: {ZIP_PATH} ({size_mb:.1f} MB)')
else:
    print('no checkpoint to zip — run the training cell first')